#**Deep Natural Language Processing @ PoliTO**

---


**Teaching Assistant:** Giuseppe Gallipoli

**Credits:** Moreno La Quatra

**Practice 2:** Word and Sentence Embeddings

## Word Embedding

![](https://qph.fs.quoracdn.net/main-qimg-3e812fd164a08f5e4f195000fecf988f)


**Key takeaways** from lessons and in-class practices:
- Word embeddings are able to map words into a semantic-aware vector space.
- There are multiple architectures for the generation of word embeddings.
- Each architecture has its advantages and disadvantages.
- Word embedding evaluation could be intrinsic (intermediate tasks) or extrinsic (downstream task).
- It is possible to use pre-trained word embedding models or use large amount of text to train it from scratch.
- The use of pre-trained word embedding models is a common practice in NLP and removes the need of training a word embedding model from scratch (that could be very time consuming and computationally expensive).

### **Question 1**

Train a new Word2Vec model using gensim with the text8 corpus available in the Python package ([reference](https://radimrehurek.com/gensim/downloader.html)). Compute the training time for the model and store it for subsequent steps.

**Hint:** you can use the following code to load the text8 corpus:

```python
import gensim.downloader as api
from gensim.models import Word2Vec
import time
dataset = api.load("text8")
```

In [1]:
! pip install --upgrade gensim


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# your code here
import gensim.downloader as api
from gensim.models import Word2Vec
import time

# Load the text8 corpus
dataset = api.load("text8")

# Start timing
start_time = time.time()

# Train a Word2Vec model
w2v_model = Word2Vec(sentences=dataset, vector_size=100, window=5, min_count=5, workers=4)

# End timing
end_time = time.time()
w2v_training_time = end_time - start_time

print(f"Word2Vec training time: {w2v_training_time:.2f} seconds")

[==================================================] 100.0% 31.6/31.6MB downloaded
Word2Vec training time: 75.36 seconds


### **Question 2**
Perform **intrinsic** evaluation of the model for the task of word analogy by exploiting the data collection available [here](https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/google_analogies.csv).

1. read CSV file
2. group analogy entries by type (column: `type`)
3. for each type of entry (**in the lab, just set type="family"** to reduce the required time) use the first 3 word vectors to compute the fourth
    - Entry: `Athens,Greece,Baghdad,Iraq`
    - `v(Greece) - v(Athens) + v(Baghdad) = res_v`
    - Get the most similar vectors to `res_v`
    - Compute in how many cases the correct word is among the top k (if `v[Iraq]` is among the k most similar words) with `k = 1, 3, 5, 10`

$top(k) = \dfrac{\sum_{i=1}^{N} f(i)}{|E|}$

where $f(i) = 1$ if the target word is among the top k and $f(i) = 0$ otherwise.

$|E|$ is the total number of entries for the considered type.

**Notes:**
1. Try with the model trained on `text8`, is there any issue? If yes, how can you solve it?
2. Test the model trained on Google News available in gensim.

In [3]:
%%capture
! wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/google_analogies.csv
! pip install --upgrade pandas

In [ ]:
# Executing this cell could take ~5 minutes
import gensim.downloader
w2v_google_news_model = gensim.downloader.load('word2vec-google-news-300')

[=========-----------------------------------------] 19.7% 327.9/1662.8MB downloaded

In [ ]:
# your code here
import pandas as pd

# 1. Read CSV file
analogies_df = pd.read_csv('google-analogies.csv')

# 2. Group analogy entries by type (we only use type="family" for the lab)
family_analogies = analogies_df[analogies_df['type'] == 'family']

# 3. Evaluation function
def evaluate_analogy(model, analogies, top_ks=[1, 3, 5, 10]):
    results = {k: 0 for k in top_ks}
    total = 0
    for idx, row in analogies.iterrows():
        # Corrected column names
        a, b, c, d = row['word1'], row['word2'], row['word3'], row['target']
        # Check if all words are in the model's vocab
        try:
            # For Gensim >=4.0, use .key_to_index
            vocab = model.key_to_index if hasattr(model, 'key_to_index') else model.wv.key_to_index
            if not all(word in vocab for word in [a, b, c, d]):
                continue
            # Compute analogy vector
            if hasattr(model, 'wv'):
                # For gensim Word2Vec/KeyedVectors
                res = model.wv.most_similar(positive=[b, c], negative=[a], topn=max(top_ks))
            else:
                # For KeyedVectors directly
                res = model.most_similar(positive=[b, c], negative=[a], topn=max(top_ks))
            predicted_words = [w for w, _ in res]
            for k in top_ks:
                if d in predicted_words[:k]:
                    results[k] += 1
            total += 1
        except Exception as e:
            # Skip analogy if any word is missing or error occurs
            continue
    # Compute top(k) for each k
    topk_scores = {k: (results[k] / total if total > 0 else 0.0) for k in top_ks}
    return topk_scores, total

# Try with the model trained on text8 (w2v_model)
print("Evaluating Word2Vec model trained on text8 (family analogies):")
w2v_scores, w2v_total = evaluate_analogy(w2v_model, family_analogies)
for k, score in w2v_scores.items():
    print(f"top({k}): {score:.3f} ({w2v_total} evaluated)")

# Note: If w2v_total is much less than the number of family analogies, it's likely due to missing words in the smaller text8 vocabulary.

# Try with the Google News model
print("\nEvaluating Word2Vec Google News model (family analogies):")
gn_scores, gn_total = evaluate_analogy(w2v_google_news_model, family_analogies)
for k, score in gn_scores.items():
    print(f"top({k}): {score:.3f} ({gn_total} evaluated)")

Evaluating Word2Vec model trained on text8 (family analogies):
top(1): 0.602 (420 evaluated)
top(3): 0.729 (420 evaluated)
top(5): 0.779 (420 evaluated)
top(10): 0.824 (420 evaluated)

Evaluating Word2Vec Google News model (family analogies):
top(1): 0.846 (506 evaluated)
top(3): 0.923 (506 evaluated)
top(5): 0.953 (506 evaluated)
top(10): 0.974 (506 evaluated)


### **Question 3**

Train a new FastText model using gensim with text8 corpus available in the Python package ([reference](https://radimrehurek.com/gensim/downloader.html)). Compute the training time for the model and store it for subsequent steps.

- Is there any significant difference in training time if compared with Word2Vec training?

In [ ]:
# your code here

### **Question 4**
Provide the same evaluation done in Question 2 for the FastText model. In this case, you can use the same type of analogy (family) and the same k values.

**Notes:**
- Try with the model trained on `text8`, is there any issue? What does it mean?
- Test the model trained on Wikipedia+News available in gensim.

In [ ]:
# Executing this cell could take ~5 minutes
import gensim.downloader
ft_wiki_news_model = gensim.downloader.load('fasttext-wiki-news-subwords-300')

In [ ]:
# your code here

### **Question 5** (optional)

Provide a complete evaluation of the best performing models (Word2Vec and FastText) by leveraging the complete dataset of analogy entries. In this case, you should use all the analogy types and all you can use the same k values provided in Question 2.

In [ ]:
# your code here

## Sentence Embeddings

Sentence embeddings are a way to represent a sentence in a vector space. The vector space is usually learned from a large corpus of text. They are used in many NLP tasks, such as text classification, text similarity, and question answering. In this practice, we will use and interact both with Doc2Vec and InferSent models.

**Key takeaways** from lessons and in-class practices:
- Doc2Vec is an extension of the Word2Vec framework.
- It incorporates Document ID to obtain a more accurate representation of a document/paragraph.
- Training document vectors are pre-computed, however you can infer vectors for new documents.
- InferSent exploit a deep learning architecture to supervisedly learn sentence representations.
- InferSent vectors could exploit both Word2Vec or FastText as word embedding models.

### **Question 6**

Train a novel Doc2Vec model using the [APIs provided by gensim](https://radimrehurek.com/gensim/models/doc2vec.html) with text8 corpus.

- Which is the training time for the model? Is it comparable with Word2Vec and FastText training time?

**Note:** Store the model to a file for subsequent steps.

In [ ]:
# your code here

### **Question 7 (Doc2Vec qualitative evaluation)**
Perform some **qualitative** experiments by computing the cosine similarities between sentences composed by yourself.
For example, you can use the following sentences:

```python
s1 = "The president of the United States is Donald Trump"
s2 = "The president of the United States is Joe Biden"
s3 = "United States is a country"
s4 = "The cell phone is a device"
```

Please try to interact with the model by providing different sentences and check the results. Is the model able to capture the semantic meaning of the sentences? Are you satisfied with the results?

In [ ]:
# your code here

### **Question 8**

Load the InferSent model provided by Facebook Research ([reference](https://github.com/facebookresearch/InferSent)) and perform the same qualitative evaluation done in Question 7. In this case, you can use the InferSent pretrained model (v2) - [reference](https://github.com/facebookresearch/InferSent).

Try to find some sentences for which InferSent is able to capture the semantic meaning of the sentences as opposed to Doc2Vec. Are you satisfied with the results? Which model is able to better capture the semantic meaning of the sentences? What can be the reason for this?

**Note:**
Please find below the code to download the InferSent model.

In [ ]:
%%capture
# InferSent download required files

! mkdir fastText
! curl -Lo fastText/crawl-300d-2M.vec.zip https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip
! unzip fastText/crawl-300d-2M.vec.zip -d fastText/
! mkdir encoder
! curl -Lo encoder/infersent2.pkl https://dl.fbaipublicfiles.com/infersent/infersent2.pkl
! git clone https://github.com/facebookresearch/InferSent.git

In [ ]:
from InferSent.models import InferSent
import torch
V = 2
MODEL_PATH = 'encoder/infersent%s.pkl' % V
params_model = {'bsize': 64, 'word_emb_dim': 300, 'enc_lstm_dim': 2048,
                'pool_type': 'max', 'dpout_model': 0.0, 'version': V}
infersent = InferSent(params_model)
infersent.load_state_dict(torch.load(MODEL_PATH))

W2V_PATH = 'fastText/crawl-300d-2M.vec'
infersent.set_w2v_path(W2V_PATH)

**Note:** Due to compatibility issues between newer NumPy versions and InferSent, you may encounter the following error when calling the `encode` method of the InferSent object:
> ValueError: setting an array element with a sequence...

If this occurs, you can fix it as follows:
- Restart the session
- Before loading the InferSent class, modify the `models.py` file in the InferSent folder by replacing line 207 with `sentences = np.array(sentences, dtype=object)[idx_sort]`

In [ ]:
# your code here

### **Question 9** (Extrinsic Evaluation)

**Extrinsic** evaluation aims at measuring the performance of the word/sentence/paragraph embedding model when used in a downstream task. In this case, we will use the model to perform a text classification task.
We can use different configuration, training corpora or even different models to build a complete architecture for the task at hand.

For this practice we use the text classification dataset available [here](https://github.com/MorenoLaQuatra/DeepNLP/blob/main/practices/P2/news_headline_classification.csv) - [source: Kaggle](https://www.kaggle.com/rmisra/news-category-dataset). It contains news headlines and the corresponding category. The dataset is composed by 200846 divided into multiple categories (e.g. politics, business, sports, etc.).

**Note:** consider using just the first 10.000 headlines to reduce runtime during the lab. You can use the complete data collection at home to achieve better results.

Compute the accuracy of 3 classification models each one built with one of the models introduced in this practice:
- Word2Vec model pretrained on Google News corpus
- FastText model pretrained on Wikipedia+News corpus
- **[Optional]** Doc2Vec model pretrained on Text8 corpus
- **[Optional]** InferSent pretrained model (v2) - [reference](https://github.com/facebookresearch/InferSent)

The procedure to create a classification system is sketched below:
1. Choose a machine learning (multi-class) classifier (e.g., MLP)
2. Split the data collection in train/test (80%/20%)
3. Use text vectors obtained by pretrained model as input of the classifier
4. Measure the accuracy of the classification system
5. Repeat step 3-4 using different embedding models


**Note:** For word embedding models you must use an aggregation strategy to obtain a single vector for each sentence. You can use the average of the word vectors or the sum of the word vectors. In both cases, the output vector can be used as input of the classifier.

Report the performance of each classification pipeline. Which model has better performance? Why? Try to elaborate on the results.

In [ ]:
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/news_headline_classification.csv

In [ ]:
# your code here

**Word2Vec + Average aggregation function**

In [ ]:
# your code here

**FastText + Average aggregation function**

In [ ]:
# your code here

**Doc2Vec (Text8)**

In [ ]:
# your code here

**InferSent**

In [ ]:
# your code here